In [0]:
%sql
-- 1. Criação dos Schemas da Arquitetura Medalhão
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Camada Bronze: Dados brutos da Steam exatamente como vieram da fonte';

CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Camada Silver: Dados limpos, tipados, deduplicados e padronizados';

CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Camada Gold: Tabelas Fato e Dimensão (Star Schema) prontas para análise de negócio';

-- 2. Criação do Volume no Unity Catalog para armazenar o arquivo CSV original
CREATE VOLUME IF NOT EXISTS workspace.bronze.arquivos_brutos
COMMENT 'Volume para armazenamento dos arquivos CSV brutos baixados do Kaggle';

In [0]:

from pyspark.sql import functions as F
import re

# Caminho do arquivo dentro do Volume do Unity Catalog
caminho_volume = "/Volumes/workspace/bronze/arquivos_brutos/games.csv"

# 1. EXTRACT: Leitura do CSV bruto tratando textos multilinhas e aspas
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false") # Na Bronze mantemos tudo como texto (String) para não perder nenhum registro por erro de conversão!
    .option("multiLine", "true")    # Essencial na base da Steam pois a coluna 'About the game' tem quebras de linha
    .option("escape", "\"")         # Evita que aspas dentro do texto quebrem as colunas
    .option("quote", "\"")
    .load(caminho_volume)
)

# 2. Adequação técnica dos nomes das colunas (Delta Lake não aceita espaços ou caracteres especiais no cabeçalho)
def padronizar_nome_coluna(nome):
    nome_limpo = re.sub(r'[^a-zA-Z0-9_]', '_', nome.strip().lower())
    return re.sub(r'_+', '_', nome_limpo)

for col_antiga in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(col_antiga, padronizar_nome_coluna(col_antiga))

# 3. Adição de Metadados de Controle da Camada Bronze (Conforme enunciado da Arquitetura Medalhão)
df_bronze = (
    df_raw
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
    .withColumn("_fonte_dados", F.lit("Kaggle - Steam Games Dataset (FronkonGames)"))
)

# 4. LOAD: Persistência como tabela gerenciada Delta Lake no Unity Catalog
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.steam_games_raw")
)

print(f"Total de registros carregados na camada Bronze: {df_bronze.count()}")
display(df_bronze.limit(10))

Total de registros carregados na camada Bronze: 125855


appid,name,release_date,estimated_owners,peak_ccu,required_age,price,discountdlc_count,about_the_game,supported_languages,full_audio_languages,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,user_score,positive,negative,score_rank,achievements,recommendations,notes,average_playtime_forever,average_playtime_two_weeks,median_playtime_forever,median_playtime_two_weeks,developers,publishers,categories,genres,tags,screenshots,movies,_data_ingestao,_arquivo_origem,_fonte_dados
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.0,0,0,null,[],[],null,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/header.jpg?t=1699268702,null,null,null,True,False,False,0,null,0,0,0,null,0,0,null,0,0,0,0,null,null,null,null,null,"https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_7d9504b958d0b143d053d31cb74b375daba338d6.1920x1080.jpg?t=1699268702,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_85d2749d23e115b4d44e8ed1024c55a6be524fde.1920x1080.jpg?t=1699268702",2026-09-25T23:43:28.880Z,dbfs:/Volumes/workspace/bronze/arquivos_brutos/games.csv,Kaggle - Steam Games Dataset (FronkonGames)
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa Prefecture, Kamakura City, for the first time in seven years, and is greeted by his older cousin Sakura Narumi (complete with maid outfit). He wanted to live in peace, but his life at the academy placed on the coastlands becomes bustling and brilliant while surrounded by a lineup of girls with booming personalities like Hotaru Amano, the sharp tongued, half-Japanese beauty, and Alice Kamishiro, the lazy witch who loves modern-science and mail-orders. Eventually, our protagonist is coaxed into joining the action committee for the academy's traditional beauty contest by his close friend. He interacts with the heroines participating in this contest who are cute but each have an idiocyncrasy or two. Fun events (with their own difficulties) pop up again and again. Your story with 'her' about smiles, peace, passion, and love, all while borrowing the power of a witch, is about to begin! 'Supipara - Alice the Magical Conductor.' is an interactive novel themed around 'dynamicism' depicted through a rallying all of the production techniques minori has accumulated to date. See expressive character portraits with moving eyes, mouths, even hair! It is quite immersive in many aspects. Also, 'Supipara' won't conclude in a single game. It will be released as several standalone episodes. Fans may freely pursue only the episodes that interest them if so desired. You could say 'it's a light novel series, and independent stories which develop their own unique personalities all in one.'",['English'],[],null,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/header.jpg?t=1725519097,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,null,0,252,3,null,0,231,null,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure,"Adventure,Visual Novel,Anime,Cute","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_0a5ff85870a1c2221e495b75a5c7b9e248eb3249.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_f53e18a6c7be8eeb39e7c8efff8395b444786f68.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_1f86a8d171a549c504acd6592f559fc4bd9bd448.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_f40b6e8e535047b60412b8f102b907cb2317d53d.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_32c2149186b260804a2a8bed8

In [0]:
%sql
COMMENT ON TABLE workspace.bronze.steam_games_raw IS 
'Tabela Bronze contendo os registros brutos de jogos da plataforma Steam sem tratamento de tipos ou limpeza, acrescida de metadados de auditoria (_data_ingestao e _arquivo_origem).';

SELECT appid, name, release_date, price, supported_languages, _data_ingestao, _arquivo_origem 
FROM workspace.bronze.steam_games_raw 
LIMIT 10;

appid,name,release_date,price,supported_languages,_data_ingestao,_arquivo_origem
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.0,null,2026-09-25T23:43:09.220Z,dbfs:/Volumes/workspace/bronze/arquivos_brutos/games.csv
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,"Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa Prefecture, Kamakura City, for the first time in seven years, and is greeted by his older cousin Sakura Narumi (complete with maid outfit). He wanted to live in peace, but his life at the academy placed on the coastlands becomes bustling and brilliant while surrounded by a lineup of girls with booming personalities like Hotaru Amano, the sharp tongued, half-Japanese beauty, and Alice Kamishiro, the lazy witch who loves modern-science and mail-orders. Eventually, our protagonist is coaxed into joining the action committee for the academy's traditional beauty contest by his close friend. He interacts with the heroines participating in this contest who are cute but each have an idiocyncrasy or two. Fun events (with their own difficulties) pop up again and again. Your story with 'her' about smiles, peace, passion, and love, all while borrowing the power of a witch, is about to begin! 'Supipara - Alice the Magical Conductor.' is an interactive novel themed around 'dynamicism' depicted through a rallying all of the production techniques minori has accumulated to date. See expressive character portraits with moving eyes, mouths, even hair! It is quite immersive in many aspects. Also, 'Supipara' won't conclude in a single game. It will be released as several standalone episodes. Fans may freely pursue only the episodes that interest them if so desired. You could say 'it's a light novel series, and independent stories which develop their own unique personalities all in one.'",2026-09-25T23:43:09.220Z,dbfs:/Volumes/workspace/bronze/arquivos_brutos/games.csv
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"Immerse yourself in the most beloved, mystical and entrancing world of Edgar Allan Poe's ''The Raven!'' This work reflected the image of the poet himself, his life and mysterious, gloomy and wonderful poetics. Explore 12 locations connected with Poe's life and work, and earn lots of cool trophies. Levels of varying complexity provide several hours of exciting gameplay every day, and special Golden Cards add an extra layer of fun. This game is for people who enjoy card games as spider solitaire, pyramid, hearts, klondike, free cell, crescent, golf, addiction, three peaks, yukon, forty thieves, scorpion, gaps, canfield, crazy quilt, algerian, montana, eight off, calculation, clock solitaire. But regardless of the type, solitaire is an age-old game that never loses its flavor! - Let your emotions run wild! - Solitaire card game – collect chains of cards! - Never a dull moment with super exciting levels! - Vivid graphics and a Pirate soundtrack make this Solitaire a win-win! - Get rid of cards faster with jokers, and increase the combo multiplier to earn more coins! - Earn 15 mystical items! - Original themed decks and 12 card back styles! - Let the fun take over!",2026-09-25T23:43:09.220Z,dbfs:/Volumes/workspace/bronze/arquivos_brutos/games.csv
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",8.99,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' It's been three years since I've repeated such greetings. Hiyoro, who couldn't avoid Hakko's broadcast, fell into a lot of skepticism, self-destruction, and debt. Now, the broadcast that I turned on with the idea of the last broadcast, with an unknown algorithm, got a lot of response, and it ended up in a mess- Hiyoru, who suddenly received everyone's attention, became a popular YouTuber... Survive the controversy, red pills, and video donne! The survival of new YouTuber Hiyoru, 'Berber YouTuber Paranoia' is starting now! the number of people produced Produced by Eugene

In [0]:
# Substitua a leitura df_raw no Notebook 01 por esta configuração:
caminho_volume = "/Volumes/workspace/bronze/arquivos_brutos/games.csv"
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("unescapedQuoteHandling", "STOP_AT_DELIMITER")
    .load(caminho_volume)
)